In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

if os.getenv("OPENAI_API_KEY") is None:
    raise ValueError("OPENAI_API_KEY is not set")

## PART 1 — PDF Ingestion & Preprocessing

**Task 1: Load PDF Documents**

1. Use LangChain PDF loaders to load PDFs.
2. Print:
   - Number of pages
   - Sample page content

In [1]:
from langchain_community.document_loaders import PyPDFLoader

/var/folders/3k/2r4fckl56zxdspgzm1tw5pdw0000gn/T/ipykernel_35351/4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/abhishekroy/Documents/tutedude/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
loader = PyPDFLoader("hp1.pdf")
data = loader.load()
print(len(data))
print(data[0])

227
page_content='/ 
THE BOY WHO LIVED 
Mr. and Mrs. Dursley, of number four, Privet Drive, 
were proud to say that they were perfectly normal, 
thank you very much. They were the last people youâ€™d 
expect to be involved in anything strange or 
mysterious, because they just didnâ€™t hold with such 
nonsense. 
Mr. Dursley was the director of a firm called 
Grunnings, which made drills. He was a big, beefy 
man with hardly any neck, although he did have a 
very large mustache. Mrs. Dursley was thin and 
blonde and had nearly twice the usual amount of 
neck, which came in very useful as she spent so 
much of her time craning over garden fences, spying 
on the neighbors. The Dursley s had a small son 
called Dudley and in their opinion there was no finer 
boy anywhere. 
The Dursleys had everything they wanted, but they 
also had a secret, and their greatest fear was that 
somebody would discover it. They didnâ€™t think they 
could bear it if anyone found out about the Potters. 
Mrs. Pott

**Task 2: Text Splitting**

1. Use RecursiveCharacterTextSplitter.
2. Configure:
   - chunk_size
   - chunk_overlap
3. Split PDF content into chunks.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(data)

print(f"Splitted into {len(chunks)} chunks")
print(f"sample chunk: {chunks[0]}")

Splitted into 668 chunks
sample chunk: page_content='/ 
THE BOY WHO LIVED 
Mr. and Mrs. Dursley, of number four, Privet Drive, 
were proud to say that they were perfectly normal, 
thank you very much. They were the last people youâ€™d 
expect to be involved in anything strange or 
mysterious, because they just didnâ€™t hold with such 
nonsense. 
Mr. Dursley was the director of a firm called 
Grunnings, which made drills. He was a big, beefy 
man with hardly any neck, although he did have a 
very large mustache. Mrs. Dursley was thin and 
blonde and had nearly twice the usual amount of 
neck, which came in very useful as she spent so 
much of her time craning over garden fences, spying 
on the neighbors. The Dursley s had a small son 
called Dudley and in their opinion there was no finer 
boy anywhere. 
The Dursleys had everything they wanted, but they 
also had a secret, and their greatest fear was that 
somebody would discover it. They didnâ€™t think they 
could bear it if anyone foun

## PART 2 — Embeddings & Vector Store

**Task 3: Create Embeddings**

1. Use any embedding model:
   - OpenAI / HuggingFace / Ollama
2. Generate embeddings for PDF chunks.

In [6]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)


**Task 4: Store Embeddings in Vector Store**
1. Store embeddings in:
   - FAISS or ChromaDB
2. Create a retriever from the vector store.

In [7]:
index = Chroma.from_documents(chunks, embeddings)
retriever = index.as_retriever()

## PART 3 — Conversational Prompt with Message History

**Task 5: RAG Prompt Template**
Create a ChatPromptTemplate that includes:

1. System message (instructions to answer from PDF only)
2. MessagesPlaceholder for chat history
3. Human message for current user question

Ensure the prompt instructs the model to:

- Use only retrieved PDF context
- Say "I don't know" if the answer is not found

In [8]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


In [9]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful AI assistant.
Answer clearly and concisely.
Use ONLY the context below. If unsure, say you don't know.

Context:
{context}
"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])


In [10]:
from operator import itemgetter

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

parser = StrOutputParser()

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

rag_chain = {
    "context": itemgetter("input") | retriever | format_docs,
    "chat_history": itemgetter("chat_history"),
    "input": itemgetter("input"),
} | prompt | llm | parser

chatbot = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)


/Users/abhishekroy/Documents/tutedude/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [11]:

cfg = {"configurable": {"session_id": "ses1"}}
print(chatbot.invoke({"input": "What is this document about?"}, config=cfg))
print("---")
print(chatbot.invoke({"input": "Can you summarize that in one line?"}, config=cfg))


The document is about a letter Harry Potter received, which includes information regarding his acceptance to Hogwarts School of Witchcraft and Wizardry. It outlines the uniform requirements and course books needed for first-year students.
---
The document discusses Harry Potter's experiences and interactions related to his acceptance and preparations for attending Hogwarts.


## PART 4 — Conversational RAG Chain

**Task 6: Build Conversational RAG Chain**
Create a pipeline:
User Question → Retriever → PDF Context → Prompt + Message History → LLM → Answer

In [12]:
prompt = ChatPromptTemplate.from_messages([
    ("system", ("""
    You are a helpful AI assistant.

    Answer the following question clearly and concisely.
    Use ONLY the context below. If unsure, say you don't know.
    Context: {context}
    """)),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

In [13]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

store = {}
def get_chat_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain = {
    "context":  itemgetter("question") | retriever | format_docs,
    "question": itemgetter("question"),
    "chat_history": itemgetter("chat_history"),
} | prompt | llm | parser





**Task 7: Maintain Message History**

1. Store user and AI messages after every turn.
2. Inject message history using MessagesPlaceholder.

In [14]:
chatbot = RunnableWithMessageHistory(
    chain,
    get_chat_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)


/Users/abhishekroy/Documents/tutedude/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [15]:
chatbot.invoke({"question": "What is the main idea of the document?"}, config={"configurable": {"session_id": "ses1"}})

"The main idea of the document revolves around Harry Potter's experiences and interactions with magical elements, particularly his discovery of the Ministry of Magic and the significance of magic in the wizarding world, as well as his relationships with other characters like Hagrid, Uncle Vernon, and his classmates. It highlights Harry's curiosity and the contrast between the magical and non-magical worlds."

**Task 8: Trimming Chat History**

1. Limit conversation history by:
   - Number of messages OR
   - Token length
2. Remove oldest messages when limits are exceeded.

In [16]:
from langchain_core.messages import trim_messages

In [17]:
trimmer = trim_messages(
    max_tokens=1000, 
    strategy="last",
    allow_partial=False,
    include_system=True,
    token_counter=llm,
    )

trimmed_chain = {
    "context":  itemgetter("question") | retriever | format_docs,
    "question": itemgetter("question"),
    "chat_history": itemgetter("chat_history") | trimmer,
} | prompt | llm | parser

In [18]:
trimmed_chatbot = RunnableWithMessageHistory(
    trimmed_chain,
    get_chat_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

/Users/abhishekroy/Documents/tutedude/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


## PART 5 — Multi-Turn Conversation Testing

**Task 9: Follow-Up Q&A Testing**

Test the chatbot with:

1. Initial factual question from the PDF
2. Follow-up question referring to previous answer
3. Clarification question

Verify:

- Context is preserved
- Answers remain grounded in the PDF

In [19]:
questions = [
    "Where does Harry live before going to Hogwarts?",
    "Who tells him that he is a wizard?",
]

cfg = {"configurable": {"session_id": "ses2"}}

for question in questions:
    result = trimmed_chatbot.invoke({"question": question}, config=cfg)
    print(f"Question: {question}")
    print(f"Answer: {result}")
    print("---")


Question: Where does Harry live before going to Hogwarts?
Answer: Harry lives with his aunt and uncle, the Dursleys, who are Muggles.
---
Question: Who tells him that he is a wizard?
Answer: Hagrid tells Harry that he is a wizard.
---


## PART 6 — Mini Project: Conversational PDF Chatbot

**Task 10: Build Final Chatbot Application**
Build a chatbot that:

- Accepts user questions
- Retrieves relevant PDF chunks
- Maintains conversation history
- Answers follow-up questions accurately
  (Optional: Streamlit UI with chat interface)


In [20]:
questions = [
    # Initial factual questions
    "Who is Harry Potter's best friend at Hogwarts?",
    "What house is Harry Potter sorted into?",
    "Who are Harry's two closest friends at Hogwarts?",
    "Who tells Harry that he is a wizard?",
    "What is the name of Harry's owl?",

    # Follow-up questions referencing previous answers
    "What house is he in?",
    "How did he become friends with Harry?",
    "Where did they first meet?",
    "Why did they become friends?",
    "What role does he play in Harry's first year at Hogwarts?",

    # Clarification / conversational context
    "Can you explain why he was important to Harry?",
    "What happened to his family?",
    "Why did Harry trust him?",
    "What was their biggest adventure together?",
    "How did they help Harry reach the Philosopher's Stone?",

    # More context-dependent questions
    "What obstacles did they face while trying to reach the Stone?",
    "Who was actually trying to get the Philosopher's Stone?",
    "Why did Harry suspect Snape?",
    "How did Harry finally discover the truth?",
    "Can you summarize everything we discussed about Harry and his friends?"
]

In [21]:
cfg = {"configurable": {"session_id": "ses3"}}

for question in questions:
    result = trimmed_chatbot.invoke({"question": question}, config=cfg)
    print(f"Question: {question}")
    print(f"Answer: {result}")
    print("---")

Question: Who is Harry Potter's best friend at Hogwarts?
Answer: Harry Potter's best friend at Hogwarts is Ron Weasley.
---
Question: What house is Harry Potter sorted into?
Answer: Harry Potter is sorted into Gryffindor.
---
Question: Who are Harry's two closest friends at Hogwarts?
Answer: Harry's two closest friends at Hogwarts are Ron Weasley and Hermione Granger.
---
Question: Who tells Harry that he is a wizard?
Answer: Hagrid tells Harry that he is a wizard.
---
Question: What is the name of Harry's owl?
Answer: Harry's owl is named Hedwig.
---
Question: What house is he in?
Answer: Harry Potter is in Gryffindor house.
---
Question: How did he become friends with Harry?
Answer: Ron Weasley became friends with Harry Potter on the Hogwarts Express when they met and shared a compartment. Ron was excited to meet Harry, who was famous, and they bonded over their experiences and shared interests.
---
Question: Where did they first meet?
Answer: Harry and Ron first met on the Hogwarts 

**Task 11: Observations & Insights**
Write short answers:

1. Difference between PDF Q&A and conversational PDF Q&A → normal PDF Q&A = one question → retrieve chunks → answer (no memory). conversational PDF Q&A also keeps chat history, so follow-ups like “what house is he in?” still resolve after talking about Harry/Ron.
2. Role of message history in follow-up questions → history carries names/entities from earlier turns into the next prompt. without it the retriever only sees a short vague follow-up and often loses the subject.
3. Trade-offs between long memory and performance → more history = better continuity, but more tokens → slower, costlier, can hit context limits, and old chatter can distract the model.
4. How trimming history affects answer quality → trimming drops oldest turns so prompts stay small/fast. recent follow-ups still work; very old details can disappear unless you summarize instead of hard-deleting.
